<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 05 · Why Failed Backtests Still Look Convincing

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

loaded_code = sys.modules.get("code")
if loaded_code is not None and not hasattr(loaded_code, "__path__"):
    del sys.modules["code"]

PROJECT_ROOT

The lab uses the same local daily market dataset as the surrounding
trading chapters.


In [ ]:
from code.labs.lab05_failed_backtests import load_returns

In [ ]:
rets = load_returns("SPY")

In [ ]:
rets.head().round(4)

## Failure Mode 1: Look-Ahead Bias
Look-ahead bias appears when the strategy uses information that would not have
been available at the decision time.


In [ ]:
from code.labs.lab05_failed_backtests import lookahead_comparison

In [ ]:
paths = lookahead_comparison("SPY")

In [ ]:
paths.head().round(4)

In [ ]:
import pandas as pd

In [ ]:
from code.labs.lab05_failed_backtests import summarize_returns

In [ ]:
summary = pd.DataFrame(
    {
        name: summarize_returns(paths[name])
        for name in paths.columns
    }
).T

In [ ]:
summary.round(3)

## Failure Mode 2: Ignoring Trading Costs
The next failure mode is more subtle.


In [ ]:
from code.labs.lab05_failed_backtests import momentum_returns

In [ ]:
mom = momentum_returns(lookback=20, cost=0.0005)

In [ ]:
mom[["position", "gross", "net", "turnover"]].tail().round(4)

In [ ]:
from code.labs.lab05_failed_backtests import cost_comparison

In [ ]:
cost_comparison(lookback=20).round(3)

## Failure Mode 3: Over-Tuning the Backtest
A third failure mode appears when many parameter choices are tested and the
best one is reported as if it had been specified in advance.


In [ ]:
from code.labs.lab05_failed_backtests import parameter_search

In [ ]:
search = parameter_search()

In [ ]:
search.round(3)

## Figure Generation (Optional)
Run the lab figure scripts under `code/figures/` to regenerate the PNG files
under `assets/figures/`.


In [ ]:
# Figure generation code adapted from `code/figures/lab05_cost_drag.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab05_failed_backtests import equity_curve
from code.labs.lab05_failed_backtests import momentum_returns

def main() -> None:
    """Generate the gross-versus-net momentum equity comparison."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    data = momentum_returns(lookback=20)
    curves = data[["gross", "net"]].apply(equity_curve)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(curves.index, curves["gross"], label="Gross")
    ax.plot(curves.index, curves["net"], label="Net of costs")
    ax.set_title("Transaction costs lower the realized path")
    ax.set_ylabel("Equity curve")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from `code/figures/lab05_lookahead_equity.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab05_failed_backtests import equity_curve
from code.labs.lab05_failed_backtests import lookahead_comparison

def main() -> None:
    """Generate the look-ahead bias equity comparison."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    curves = lookahead_comparison().apply(equity_curve)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for col in curves.columns:
        ax.plot(curves.index, curves[col], label=col)
    ax.set_yscale("log")
    ax.set_title("Look-ahead bias can dominate the whole result")
    ax.set_ylabel("Equity curve, log scale")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from `code/figures/lab05_parameter_search.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

ROOT = PROJECT_ROOT

from code.labs.lab05_failed_backtests import parameter_search

def main() -> None:
    """Generate the parameter-search comparison figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    results = parameter_search()
    xpos = np.arange(len(results))
    width = 0.38

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.bar(
        xpos - width / 2,
        results["train_sharpe"],
        width,
        label="Train",
    )
    ax.bar(
        xpos + width / 2,
        results["test_sharpe"],
        width,
        label="Test",
    )
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xticks(xpos)
    ax.set_xticklabels(results["lookback"].astype(str))
    ax.set_xlabel("Momentum lookback")
    ax.set_ylabel("Annualized Sharpe ratio")
    ax.set_title("Parameter search can pick unstable winners")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    ax.legend()

    fig.tight_layout()

main()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
